In [7]:
import pandas as pd

catalog_df = pd.read_xml("data/catalog.xml", parser="etree")
catalog_df = catalog_df.set_index("id")
catalog_df

,name,category,price,availability,description
id,,,,,
1,Lenovo IdeaPad 3 Laptop,Electronics,599.9,In stock,"15.6"" Full HD, AMD Ryzen 5, 8GB RAM, 512GB SSD"
2,Samsung Galaxy S24 Smartphone,Electronics,899.9,Pre-order,"6.1"" AMOLED, 128GB storage, 5G, triple camera"
3,Sony WH-1000XM5 Headphones,Audio,349.9,In stock,"Wireless, noise cancelling, up to 30 hours bat..."


In [8]:
import xml.dom.minidom

dom = xml.dom.minidom.parse("data/catalog.xml")

products = []
for product_node in dom.getElementsByTagName("product"):
    product_id = product_node.attributes["id"].value
    name = product_node.getElementsByTagName("name")[0].childNodes[0].data
    category = product_node.getElementsByTagName("category")[0].childNodes[0].data
    price_node = product_node.getElementsByTagName("price")[0]
    price = price_node.childNodes[0].data
    currency = price_node.attributes["currency"].value
    availability = product_node.getElementsByTagName("availability")[0].childNodes[0].data
    description = product_node.getElementsByTagName("description")[0].childNodes[0].data

    products.append({
        "id": product_id,
        "name": name,
        "category": category,
        "price": price,
        "currency": currency,
        "availability": availability,
        "description": description,
    })

products

[{'id': '1',
  'name': 'Lenovo IdeaPad 3 Laptop',
  'category': 'Electronics',
  'price': '599.90',
  'currency': 'USD',
  'availability': 'In stock',
  'description': '15.6" Full HD, AMD Ryzen 5, 8GB RAM, 512GB SSD'},
 {'id': '2',
  'name': 'Samsung Galaxy S24 Smartphone',
  'category': 'Electronics',
  'price': '899.90',
  'currency': 'USD',
  'availability': 'Pre-order',
  'description': '6.1" AMOLED, 128GB storage, 5G, triple camera'},
 {'id': '3',
  'name': 'Sony WH-1000XM5 Headphones',
  'category': 'Audio',
  'price': '349.90',
  'currency': 'USD',
  'availability': 'In stock',
  'description': 'Wireless, noise cancelling, up to 30 hours battery life'}]

In [9]:
import requests

url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 50.4501,
    "longitude": 30.5234,
    "daily": "temperature_2m_max,temperature_2m_min,temperature_2m_mean",
    "past_days": 31,
    "timezone": "Europe/Kyiv"
}

result = requests.get(url, params=params)
print(result.status_code)
data = result.json()
data

200


{'latitude': 50.4375,
 'longitude': 30.5,
 'generationtime_ms': 0.1323223114013672,
 'utc_offset_seconds': 10800,
 'timezone': 'Europe/Kyiv',
 'timezone_abbreviation': 'GMT+3',
 'elevation': 155.0,
 'daily_units': {'time': 'iso8601',
  'temperature_2m_max': '°C',
  'temperature_2m_min': '°C',
  'temperature_2m_mean': '°C'},
 'daily': {'time': ['2026-08-13',
   '2026-08-14',
   '2026-08-15',
   '2026-08-16',
   '2026-08-17',
   '2026-08-18',
   '2026-08-19',
   '2026-08-20',
   '2026-08-21',
   '2026-08-22',
   '2026-08-23',
   '2026-08-24',
   '2026-08-25',
   '2026-08-26',
   '2026-08-27',
   '2026-08-28',
   '2026-08-29',
   '2026-08-30',
   '2026-08-31',
   '2026-09-01',
   '2026-09-02',
   '2026-09-03',
   '2026-09-04',
   '2026-09-05',
   '2026-09-06',
   '2026-09-07',
   '2026-09-08',
   '2026-09-09',
   '2026-09-10',
   '2026-09-11',
   '2026-09-12',
   '2026-09-13',
   '2026-09-14',
   '2026-09-15',
   '2026-09-16',
   '2026-09-17',
   '2026-09-18',
   '2026-09-19'],
  'tempera

In [10]:
import datetime

weather_df = pd.DataFrame(data["daily"])
weather_df["time"] = pd.to_datetime(weather_df["time"])

today = pd.Timestamp(datetime.date.today())
weather_df = weather_df[weather_df["time"] <= today].reset_index(drop=True)

weather_df

,time,temperature_2m_max,temperature_2m_min,temperature_2m_mean
0,2026-08-13,23.1,12.9,18.6
1,2026-08-14,22.8,12.4,18.4
2,2026-08-15,27.0,13.9,21.1
3,2026-08-16,30.3,17.8,24.3
4,2026-08-17,30.6,21.1,24.7
5,2026-08-18,29.5,15.3,21.7
6,2026-08-19,22.9,13.0,17.8
7,2026-08-20,31.1,15.7,23.4
8,2026-08-21,33.5,21.0,26.9
9,2026-08-22,33.7,21.1,27.0


In [14]:
import plotly.express as px

fig = px.line(
    weather_df,
    x="time",
    y=["temperature_2m_max", "temperature_2m_min", "temperature_2m_mean"],
    markers=True,
    title="Температура в Києві за останній місяць",
    labels={
        "time": "Дата",
        "value": "Температура, °C",
        "variable": "Показник",
        "temperature_2m_max": "Максимальна",
        "temperature_2m_min": "Мінімальна",
        "temperature_2m_mean": "Середня",
    }
)
name_map = {
    "temperature_2m_max": "Максимальна",
    "temperature_2m_min": "Мінімальна",
    "temperature_2m_mean": "Середня",
}

fig.for_each_trace(lambda trace: trace.update(name=name_map[trace.name]))
fig.show()